# Détection de déforestation - démonstration de pipeline

Ce notebook montre un pipeline cohérent pour :
- charger deux images temporelles,
- extraire des représentations RGB / HSV / ratio vert,
- calculer des features locales,
- choisir le nombre de clusters par CAH,
- segmenter avec K-means,
- identifier automatiquement la végétation,
- nettoyer les masques,
- quantifier la surface végétale et la perte entre t0 et t1.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import data, util
from src.forest_pipeline import (
    rgb_to_hsv,
    green_ratio,
    build_pixel_features,
    sample_pixel_features,
    choose_cluster_count,
    segment_kmeans,
    identify_vegetation_cluster,
    clean_mask,
    compute_area,
    deforestation_map,
    overlay_mask,
    color_segments,
)

img0 = util.img_as_float(data.astronaut())
img1 = img0.copy()
mask_def = np.zeros(img1.shape[:2], dtype=bool)
mask_def[140:320, 20:260] = True
img1[mask_def, 1] *= 0.35
img1[mask_def, 2] *= 0.35
img1 = np.clip(img1, 0.0, 1.0)

In [ ]:
hsv0 = rgb_to_hsv(img0)
ratio0 = green_ratio(img0)
hsv1 = rgb_to_hsv(img1)
ratio1 = green_ratio(img1)

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes[0, 0].imshow(img0)
axes[0, 0].set_title('t0 - originale')
axes[0, 1].imshow(hsv0[..., 0], cmap='hsv')
axes[0, 1].set_title('t0 - teinte (H)')
axes[0, 2].imshow(ratio0, cmap='Greens')
axes[0, 2].set_title('t0 - ratio vert')
axes[1, 0].imshow(img1)
axes[1, 0].set_title('t1 - modifiée')
axes[1, 1].imshow(hsv1[..., 0], cmap='hsv')
axes[1, 1].set_title('t1 - teinte (H)')
axes[1, 2].imshow(ratio1, cmap='Greens')
axes[1, 2].set_title('t1 - ratio vert')
for ax in axes.ravel():
    ax.axis('off')
plt.tight_layout()

In [ ]:
features0 = build_pixel_features(img0, include_local=False)
sampled = sample_pixel_features(features0, n_samples=2500)
scores = choose_cluster_count(sampled, k_min=2, k_max=6)
print('Scores de silhouette par nombre de clusters (CAH) :')
print(scores)

labels0, _ = segment_kmeans(img0, n_clusters=4, include_local=True, radius=7)
labels1, _ = segment_kmeans(img1, n_clusters=4, include_local=True, radius=7)
seg0 = color_segments(labels0)
seg1 = color_segments(labels1)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(seg0)
axes[0].set_title('Segmentation K-means t0')
axes[1].imshow(seg1)
axes[1].set_title('Segmentation K-means t1')
for ax in axes:
    ax.axis('off')
plt.tight_layout()

In [ ]:
veg0, cluster0, cluster_scores0 = identify_vegetation_cluster(labels0, img0, 4)
veg1, cluster1, cluster_scores1 = identify_vegetation_cluster(labels1, img1, 4)
veg0 = clean_mask(veg0, opening_radius=3, closing_radius=3, min_size=300)
veg1 = clean_mask(veg1, opening_radius=3, closing_radius=3, min_size=300)
areas = [compute_area(veg0), compute_area(veg1)]
diff = deforestation_map(veg0, veg1)
overlay0 = overlay_mask(img0, veg0, color_rgb=(0.0, 1.0, 0.0), alpha=0.3)
overlay1 = overlay_mask(img1, veg1, color_rgb=(0.0, 1.0, 0.0), alpha=0.3)
overlay_loss = overlay_mask(img1, diff['lost'], color_rgb=(1.0, 0.0, 0.0), alpha=0.45)

print('Cluster végétation t0 :', cluster0, 'scores:', cluster_scores0)
print('Cluster végétation t1 :', cluster1, 'scores:', cluster_scores1)
print('Surface végétale t0 :', areas[0])
print('Surface végétale t1 :', areas[1])
print('Surface perdue :', compute_area(diff['lost']))

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
axes[0].imshow(overlay0)
axes[0].set_title('Végétation t0')
axes[1].imshow(overlay1)
axes[1].set_title('Végétation t1')
axes[2].imshow(overlay_loss)
axes[2].set_title('Perte de végétation (rouge)')
for ax in axes:
    ax.axis('off')
plt.tight_layout()